In [ ]:
# Check PyTorch installation and GPU availability
print(f"✅ PyTorch Imported Successfully")
print(f"Version: {torch.__version__}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print(f"🚀 GPU Available: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ GPU Not Available (Running on CPU)")

In [ ]:
# Load Data
# Assuming the file path is the same as in the TF version
try:
    df = pd.read_csv("data/HTRU_2.csv")  # Try local path first
except FileNotFoundError:
    try:
        df = pd.read_csv(
            "/kaggle/input/group-project-comp4432/kaggle1/HTRU_2.csv"
        )  # Fallback to kaggle path
    except FileNotFoundError:
        # Create a dummy dataset if file not found for demonstration purposes (Optional, but good for robustness)
        print("Dataset not found. Please ensure 'HTRU_2.csv' is in 'data/' folder.")
        raise

df.columns = [
    "mean_profile",
    "std_profile",
    "kurtosis_profile",
    "skewness_profile",
    "mean_dmsc",
    "std_dmsc",
    "kurtosis_dmsc",
    "skewness_dmsc",
    "target",
]
df.head()

In [ ]:
# Data Preprocessing
features = df.iloc[:, 0:8]
target = df.iloc[:, 8]

scaler = RobustScaler()
X_scaled = scaler.fit_transform(features)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, target, test_size=0.35, stratify=target, random_state=42
)

# Convert to PyTorch Tensors
X_train_tensor = torch.FloatTensor(X_train).to(device)
y_train_tensor = torch.FloatTensor(y_train.values).unsqueeze(1).to(device)
X_test_tensor = torch.FloatTensor(X_test).to(device)
y_test_tensor = torch.FloatTensor(y_test.values).unsqueeze(1).to(device)

# Create DataLoaders
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [ ]:
# Define PyTorch Model
class PulsarClassifier(nn.Module):
    def __init__(
        self, input_dim=8, hidden_dim=16, num_heads=4, ff_dim=32, dropout_rate=0.25
    ):
        super(PulsarClassifier, self).__init__()

        # Feature projection
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout_rate)
        )

        # Transformer Encoder Layer
        # batch_first=True means input shape is (batch, seq, feature)
        # Here we treat the feature vector as a sequence of length 1?
        # Or we treat the hidden_dim as the embedding dim.
        # In the TF code: MultiHeadAttention(key_dim=..., attention_axes=1)(inputs, inputs)
        # TF's MultiHeadAttention with attention_axes=1 applies attention over the feature dimension if inputs are 2D (batch, features).
        # This is a bit unusual for standard Transformers which usually operate on sequences.
        # However, to replicate the logic:
        # If input is (batch, 16), TF treats it as (batch, 16) and attention is applied.
        # Let's use a standard TransformerEncoderLayer but we need to reshape input to (batch, 1, hidden_dim) or similar if we want sequence attention.
        # BUT, looking at the TF code: `attention_axes=1` suggests it's attending across the feature dimension?
        # Actually, for 2D inputs (batch, dim), TF MHA with attention_axes=1 is self-attention over the features.
        # In PyTorch, nn.MultiheadAttention expects (seq_len, batch, embed_dim) or (batch, seq_len, embed_dim).
        # To mimic "attention over features", we can treat the 16 features as a sequence of length 16 with embedding dim 1? No, that's weird.
        # Or sequence length 1 with embedding dim 16? Then attention is trivial (it attends to itself).

        # Let's look closer at the TF code:
        # inputs shape: (batch, 16)
        # MultiHeadAttention(..., attention_axes=1)(inputs, inputs)
        # If attention_axes=1, it calculates attention scores between elements of dimension 1.
        # So it treats the 16 features as a sequence of length 16.
        # So we need to reshape (batch, 16) -> (batch, 16, 1) to use PyTorch's MHA?
        # Or just use a standard Linear layer which is what self-attention over features mostly amounts to if there's no positional info?
        # Let's try to be faithful.
        # We will reshape (batch, 16) -> (batch, 16, 1)
        # seq_len=16, embed_dim=1.

        self.hidden_dim = hidden_dim
        self.transformer_encoder = nn.TransformerEncoderLayer(
            d_model=1,  # We will treat each feature as a token of dim 1
            nhead=1,  # With d_model=1, we can only have 1 head
            dim_feedforward=ff_dim,
            dropout=dropout_rate,
            batch_first=True,
        )
        # Wait, the TF code had num_heads=4, key_dim=16//4=4.
        # This implies d_model was 16.
        # If d_model is 16 and we have 2D input (batch, 16), then seq_len must be 1.
        # If seq_len is 1, attention is a no-op (softmax([0]) = 1).
        # UNLESS `attention_axes` does something else.
        # In TF, if you pass 2D input to MHA, it usually complains unless you specify axes.
        # If it treats dim 1 as the sequence, then it's (batch, 16, 1).

        # Let's stick to a standard Transformer usage where we project to a sequence.
        # But to strictly match the TF architecture which outputs (batch, 16):
        # It seems the TF model was effectively doing a complex non-linear transformation.
        # Let's implement a standard Transformer block on the embedding.

        self.transformer_block = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=dropout_rate,
            batch_first=True,
        )

        self.final_layer = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x: (batch, 8)
        x = self.input_proj(x)  # (batch, 16)

        # Reshape for Transformer: (batch, seq_len=1, d_model=16)
        # This makes it a "sequence" of 1 token with 16 dimensions.
        # Attention here is self-attention of that 1 token (trivial).
        # But the FeedForward part of the Transformer block is still active.
        x = x.unsqueeze(1)

        x = self.transformer_block(x)

        x = x.squeeze(1)  # (batch, 16)

        out = self.final_layer(x)
        return self.sigmoid(out)


model = PulsarClassifier().to(device)
print(model)

In [ ]:
# Training Setup
criterion = nn.BCELoss()  # Binary Cross Entropy
# Class weights: 0: 1.0, 1: 2.0
# We can implement this by using a weighted loss or just passing weight to BCELoss (but BCELoss weight is per-batch element).
# Better to use BCEWithLogitsLoss with pos_weight if we removed the sigmoid, but we have sigmoid.
# Let's implement manual weighting in the loop or use a custom loss function.


def weighted_binary_crossentropy(output, target, weights=None):
    if weights is not None:
        assert len(weights) == 2

        loss = -weights[1] * (target * torch.log(output + 1e-7)) - weights[0] * (
            (1 - target) * torch.log(1 - output + 1e-7)
        )
    else:
        loss = -(target * torch.log(output + 1e-7)) - (
            (1 - target) * torch.log(1 - output + 1e-7)
        )

    return torch.mean(loss)


optimizer = optim.Adam(model.parameters(), lr=0.001)


# Early Stopping Class
class EarlyStopping:
    def __init__(self, patience=10, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
        self.best_model_state = None

    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.best_model_state = model.state_dict()
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.best_model_state = model.state_dict()
            self.counter = 0


early_stopper = EarlyStopping(patience=10)

In [ ]:
# Training Loop
epochs = 100
history = {"accuracy": [], "val_accuracy": [], "loss": [], "val_loss": []}
class_weights = {0: 1.0, 1: 2.0}

print("Starting Training...")
for epoch in range(epochs):
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)

        # Custom weighted loss
        loss = weighted_binary_crossentropy(outputs, labels, weights=class_weights)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        predicted = (outputs > 0.5).float()
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    avg_train_loss = train_loss / len(train_loader)
    train_acc = train_correct / train_total

    # Validation
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)
            loss = weighted_binary_crossentropy(outputs, labels, weights=class_weights)
            val_loss += loss.item()
            predicted = (outputs > 0.5).float()
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    avg_val_loss = val_loss / len(test_loader)
    val_acc = val_correct / val_total

    history["accuracy"].append(train_acc)
    history["val_accuracy"].append(val_acc)
    history["loss"].append(avg_train_loss)
    history["val_loss"].append(avg_val_loss)

    print(
        f"Epoch {epoch + 1}/{epochs} - Loss: {avg_train_loss:.4f} - Acc: {train_acc:.4f} - Val Loss: {avg_val_loss:.4f} - Val Acc: {val_acc:.4f}"
    )

    early_stopper(avg_val_loss, model)
    if early_stopper.early_stop:
        print("Early stopping triggered")
        model.load_state_dict(early_stopper.best_model_state)
        break

In [ ]:
# Visualization
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history["accuracy"], label="Train")
plt.plot(history["val_accuracy"], label="Validation")
plt.title("Model Accuracy")
plt.ylabel("Accuracy")
plt.xlabel("Epoch")
plt.legend(loc="lower right")

plt.subplot(1, 2, 2)
plt.plot(history["loss"], label="Train")
plt.plot(history["val_loss"], label="Validation")
plt.title("Model Loss")
plt.ylabel("Loss")
plt.xlabel("Epoch")
plt.legend(loc="upper right")

plt.show()

In [ ]:
# Evaluation
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        outputs = model(inputs)
        predicted = (outputs > 0.5).float()
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
cm_df = pd.DataFrame(
    cm, index=["Non-Pulsar", "Pulsar"], columns=["Non-Pulsar", "Pulsar"]
)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_df, annot=True, fmt="d", cbar=False)
plt.title("Confusion Matrix")
plt.ylabel("True Value")
plt.xlabel("Predicted Value")
plt.show()

# Metrics
ptbl = PrettyTable()
ptbl.field_names = ["Accuracy", "Recall", "F1Score"]
ptbl.add_row(
    [
        accuracy_score(all_labels, all_preds),
        recall_score(all_labels, all_preds),
        f1_score(all_labels, all_preds),
    ]
)
print(ptbl)

In [ ]:
# Gradio Interface
def predict(*features):
    features_array = list(features)
    # Scale features
    scaled = scaler.transform([features_array])
    # Convert to tensor
    tensor_input = torch.FloatTensor(scaled).to(device)

    model.eval()
    with torch.no_grad():
        prob = model(tensor_input).item()

    return {"Non-Pulsar": 1 - prob, "Pulsar": prob}


demo = gr.Interface(
    fn=predict,
    inputs=[gr.Number(label=col) for col in df.columns[:-1]],
    outputs=gr.Label(label="Prediction Result"),
    title="Pulsar Identification System (PyTorch)",
    description="Input eight feature values to predict if the celestial object is a pulsar",
    allow_flagging="never",
)

# demo.launch() # Uncomment to launch

In [ ]:
# Noise Sensitivity Analysis
print("\n--- Noise Sensitivity Analysis ---")
noise_levels = [0.0, 0.1, 0.2, 0.5, 1.0]
results_noise = {"Noise Level": [], "Accuracy": [], "Recall": [], "F1Score": []}

model.eval()
with torch.no_grad():
    for std_dev in noise_levels:
        print(f"\nPredicting with Noise Level (Std Dev): {std_dev}")

        # Add noise to test set
        noise = np.random.normal(0, std_dev, X_test.shape)
        X_test_noisy = X_test + noise
        X_test_noisy_tensor = torch.FloatTensor(X_test_noisy).to(device)

        # Predict
        outputs = model(X_test_noisy_tensor)
        predicted = (outputs > 0.5).float().cpu().numpy()

        # Metrics
        acc = accuracy_score(y_test, predicted)
        rec = recall_score(y_test, predicted)
        f1 = f1_score(y_test, predicted)

        results_noise["Noise Level"].append(std_dev)
        results_noise["Accuracy"].append(acc)
        results_noise["Recall"].append(rec)
        results_noise["F1Score"].append(f1)

        ptbl_noise = PrettyTable()
        ptbl_noise.field_names = ["Accuracy", "Recall", "F1-Score"]
        ptbl_noise.add_row([acc, rec, f1])
        print(ptbl_noise)

# Visualize
results_df_noise = pd.DataFrame(results_noise)
plt.figure(figsize=(10, 6))
plt.plot(
    results_df_noise["Noise Level"],
    results_df_noise["Accuracy"],
    marker="o",
    label="Accuracy",
)
plt.plot(
    results_df_noise["Noise Level"],
    results_df_noise["Recall"],
    marker="s",
    label="Recall",
)
plt.plot(
    results_df_noise["Noise Level"],
    results_df_noise["F1Score"],
    marker="^",
    label="F1-Score",
)

plt.title("Model Robustness Analysis (PyTorch)")
plt.xlabel("Noise Magnitude (Standard Deviation)")
plt.ylabel("Performance Metric Value")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.7)
plt.show()